### Taginfo

Tanken att få en översikt vilka taggar som finns på hundrastplatser i Sverige i OSM 

* Issue [#64](https://github.com/salgo60/Dogpark_Sweden/issues/64)
* Denna notebook [64_taginfo](https://github.com/salgo60/Dogpark_Sweden/tree/main/notebook/64_taginfo.ipynb)

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2025-11-12 19:43


In [3]:
import requests
import json
import os
import time

# 🗺️ Overpass API
overpass_url = "https://overpass-api.de/api/interpreter"

# 🔍 Fråga – alla hundrastgårdar i Sverige
overpass_query = """
[out:json][timeout:60];
area["ISO3166-1"="SE"][admin_level=2]->.sweden;
nwr["leisure"="dog_park"](area.sweden);
out center tags;
"""

# 💾 Cacheinställningar
cache_file = "osm_dogparks_cache.json"
cache_ttl = 60 * 60 * 24 * 7  # 7 dagar

def get_osm_dogparks():
    if os.path.exists(cache_file):
        age = time.time() - os.path.getmtime(cache_file)
        if age < cache_ttl:
            print("📦 Läser hundrastgårdar från cache...")
            with open(cache_file, "r", encoding="utf-8") as f:
                return json.load(f)
        else:
            print("⚠️ Cache äldre än 7 dagar — hämtar ny data...")

    print("🌍 Hämtar hundrastgårdar från OSM...")
    response = requests.get(overpass_url, params={"data": overpass_query})
    response.raise_for_status()
    data = response.json()

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"✅ Sparade {len(data['elements'])} objekt till cache.")
    return data

# 🚀 Kör
osm_data = get_osm_dogparks()
print(f"🐾 Hittade {len(osm_data['elements'])} hundrastgårdar.")


📦 Läser hundrastgårdar från cache...
🐾 Hittade 554 hundrastgårdar.


🐶 Antal hundrastgårdar i data: 554

📊 Vanligaste taggar (key):

                  key  count
0             leisure    554
1                name    249
2             barrier    186
3             website    136
4          wheelchair    133
5            operator    118
6              access    108
7       operator:type     87
8                 dog     76
9             surface     68
10         fence_type     65
11  operator:wikidata     64
12  wikimedia_commons     62
13      opening_hours     60
14        description     49

✅ Filer sparade:
 - taginfo_dogparks_keys.csv (antal per key)
 - taginfo_dogparks_key_value.csv (antal per key+value)

🔒 fenced=* fördelning:
         key value  count
141  fenced   yes     46
142  fenced    no      1

🚪 access=* fördelning:
       key      value  count
0  access        yes     99
1  access    private      4
2  access  customers      3
3  access     permit      2

📸 62 objekt har wikimedia_commons-länk.


In [2]:
from datetime import datetime 
# End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))


Date: 2025-11-12 19:43:26
Total time elapsed: 00 minutes 25.02 seconds
